# Audit of the original Infiltration results (Colab)

Reproduces **inference** without training or treating saved result CSVs as a substitute for inference. It uses the original graphs and checkpoints in `MyDrive/nids-mitre`. Earlier CSVs serve only as comparison references. Corrected labels are not mixed into this audit.

Run the cells in order. Drive must contain `dataset_processed`, `dataset_processed_thu0103`, and `results_earlystopping`. The code is pinned to a commit on `feat/fair-comparison-experiments`; if the remote commit changes, the first cell stops the audit to prevent silently mixing versions.


In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('torch_geometric') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)


In [3]:
from pathlib import Path
import subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/nids-mitre')
CLONE_DIR = Path('/content/temporalgnn-nids-audit')
EXPECTED_COMMIT = '2262b53d155f9e8cf5221516f47e4170e6cd0d6e'
BRANCH = 'feat/fair-comparison-experiments'
assert DRIVE_ROOT.is_dir(), f'Cannot find {DRIVE_ROOT}'
if not CLONE_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/tatipar/temporalgnn-nids.git', str(CLONE_DIR)], check=True)

subprocess.run(
    ['git', '-C', str(CLONE_DIR), 'fetch', '--depth', '1', 'origin', EXPECTED_COMMIT],
    check=True,
)
subprocess.run(
    ['git', '-C', str(CLONE_DIR), 'checkout', '--detach', EXPECTED_COMMIT],
    check=True,
)
commit = subprocess.check_output(['git', '-C', str(CLONE_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert commit == EXPECTED_COMMIT, (
    f'Unexpected code commit: {commit}. Review the original commit before inference; expected {EXPECTED_COMMIT}.')
sys.path.insert(0, str(CLONE_DIR / 'code/python'))
print('Code:', BRANCH, commit)
print('Data:', DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: feat/fair-comparison-experiments 2262b53d155f9e8cf5221516f47e4170e6cd0d6e
Data: /content/drive/MyDrive/nids-mitre


In [4]:
import glob, hashlib, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from IPython.display import display
from utils.datasets import NF_IDS_Dataset
from utils.models import SimpleMLP, E_GraphSAGE, EdgeGRU_Baseline_NoX, StaticGNN_Identity, ST_GNN_Identity
from utils.training import evaluate as infer
from utils.metrics import calculate_metrics_gnn

RESULTS = DRIVE_ROOT / 'results_earlystopping'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = {42, 123, 777, 2024, 99}
CHAMPIONS_REPORTED = {
    'SimpleMLP_BiasOn': 777,
    'EGraphSAGE_BiasOn': 99,
    'EdgeGRU_NoX_BiasOn': 99,
    'StaticGNN_BiasOn_robust_Identity': 42,
    'ST_GNN_BiasOn_robust_Identity_clone': 42,
}
BASE_PARAMS = dict(node_dim=16, edge_dim=32, hidden_dim=32,
                   dropout=0.2, output_bias_init=-2.9968)
EXPERIMENTS = [
    ('Simple MLP', 'SimpleMLP_BiasOn', SimpleMLP, False),
    ('E-GraphSAGE', 'EGraphSAGE_BiasOn', E_GraphSAGE, False),
    ('Edge GRU', 'EdgeGRU_NoX_BiasOn', EdgeGRU_Baseline_NoX, True),
    ('Static GNN', 'StaticGNN_BiasOn_robust_Identity', StaticGNN_Identity, False),
    ('ST-GNN', 'ST_GNN_BiasOn_robust_Identity_clone', ST_GNN_Identity, True),
]
SPLITS = {
    'test1': (DRIVE_ROOT / 'dataset_processed', 'test', 'test1_results'),
    'test2': (DRIVE_ROOT / 'dataset_processed_thu0103', 'test2', 'test2_results'),
}
EXPECTED_SNAPSHOTS = {'test1': 429, 'test2': 2822}
for tag, (root, split, _) in SPLITS.items():
    assert (root / split).is_dir(), f'Missing graphs for {tag}: {root / split}'
assert RESULTS.is_dir(), f'Missing results: {RESULTS}'
print('Device:', DEVICE)
print('Original experiments:', [e[1] for e in EXPERIMENTS])


Device: cpu
Original experiments: ['SimpleMLP_BiasOn', 'EGraphSAGE_BiasOn', 'EdgeGRU_NoX_BiasOn', 'StaticGNN_BiasOn_robust_Identity', 'ST_GNN_BiasOn_robust_Identity_clone']


## 1. Artifact inventory and provenance

This cell requires one `run_id` per seed in the Test 1 CSV and locates the exact checkpoint with that identifier. It checks the Test 2 champion's `run_id` against Test 1, and reports duplicates, threshold disagreements, and missing files before loading any model. It does **not** use `gather_metrics()`, which also scans older directories such as `EGraphSAGE_BiasOn_OLD`.


In [5]:
def csv_reference(tag, exp):
    suffix = 'test1_results' if tag == 'test1' else 'test2_results'
    path = RESULTS / suffix / f'{tag}_metrics_{exp}.csv'
    assert path.is_file(), f'Missing reference {path}'
    df = pd.read_csv(path)
    assert {'seed', 'run_id', 'model_name'}.issubset(df.columns), f'Missing required columns in {path}'
    df['seed'] = df['seed'].astype(int)
    assert not df['run_id'].isna().any(), f'Empty run_id in {path}'
    assert df['seed'].is_unique, f'Duplicate seeds in {path}'
    assert df['run_id'].is_unique, f'Duplicate run_id in {path}'
    if tag == 'test1':
        assert set(df['seed']) == SEEDS, f'Unexpected seeds in {path}: {set(df["seed"])}'
    else:
        assert len(df) == 1, f'Test 2 should have one champion in {path}'
        assert int(df.iloc[0]['seed']) == CHAMPIONS_REPORTED[exp], f'Champion changed in {path}'
    return df

def training_row(exp, run_id, seed):
    path = RESULTS / 'logs' / exp / f'run_metrics_{exp}.csv'
    assert path.is_file(), f'Missing training log {path}'
    df = pd.read_csv(path)
    if 'run_id' not in df.columns and 'extra_run_id' in df.columns:
        df = df.rename(columns={'extra_run_id': 'run_id'})
    assert 'run_id' in df.columns, f'Missing run_id/extra_run_id in {path}'
    matches = df[df['run_id'].astype(str) == str(run_id)]
    assert len(matches) == 1, f'{run_id}: {len(matches)} matches in {path}'
    row = matches.iloc[0]
    assert int(row['seed']) == seed, f'Seed mismatch for {run_id}'
    assert row['model_name'] == f'{exp}_seed{seed}', f'Model mismatch for {run_id}'
    return row

def checkpoint(exp, run_id):
    hits = list((RESULTS / 'saved_models' / exp).glob(f'{run_id}_*.pth'))
    assert len(hits) == 1, f'{run_id}: {len(hits)} checkpoints: {hits}'
    return hits[0]

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

references, inventory = {}, []
for display_name, exp, model_class, temporal in EXPERIMENTS:
    t1, t2 = csv_reference('test1', exp), csv_reference('test2', exp)
    references[exp] = {'test1': t1, 'test2': t2}
    winner = t2.iloc[0]
    corresponding = t1[t1['seed'] == int(winner['seed'])].iloc[0]
    assert str(winner['run_id']) == str(corresponding['run_id']), f'Test 2 uses a different run_id: {exp}'
    threshold_path = RESULTS / 'logs' / exp / f'thresholds_{exp}.npz'
    assert threshold_path.is_file(), f'Missing thresholds {threshold_path}'
    with np.load(threshold_path) as thresholds:
        assert set(thresholds.files) >= {f'seed_{s}' for s in SEEDS}, f'Incomplete seeds in {threshold_path}'
        for _, row in t1.iterrows():
            seed, run_id = int(row['seed']), str(row['run_id'])
            logged = training_row(exp, run_id, seed)
            pth = checkpoint(exp, run_id)
            threshold = float(thresholds[f'seed_{seed}'])
            cached_th = float(row['optimal_threshold']) if 'optimal_threshold' in row and pd.notna(row['optimal_threshold']) else np.nan
            inventory.append(dict(model=display_name, experiment=exp, seed=seed,
                                  run_id=run_id, checkpoint=str(pth),
                                  checkpoint_sha256=sha256(pth), threshold_npz=threshold,
                                  threshold_cached_row=cached_th,
                                  threshold_agrees=(np.isnan(cached_th) or np.isclose(threshold, cached_th, atol=1e-8)),
                                  is_test2_champion=(seed == int(winner['seed'])),
                                  training_log_model=logged['model_name']))
inventory = pd.DataFrame(inventory)
display(inventory[['model','seed','run_id','threshold_npz','threshold_cached_row','threshold_agrees','is_test2_champion']])
print('Identified checkpoints:', len(inventory))
print('Differences between .npz threshold and CSV column:', int((~inventory.threshold_agrees).sum()))

# Reproduce 1-SD selection from the five validation runs of this exact experiment.
selection = []
for _, exp, _, _ in EXPERIMENTS:
    log_dir = RESULTS / 'logs' / exp
    validation_path = log_dir / f'metrics_newth_{exp}.csv'
    if not validation_path.is_file():
        validation_path = log_dir / f'run_metrics_{exp}.csv'
    val = pd.read_csv(validation_path)
    if 'run_id' not in val.columns and 'extra_run_id' in val.columns:
        val = val.rename(columns={'extra_run_id': 'run_id'})
    assert {'seed', 'run_id', 'F1'}.issubset(val.columns), f'Missing columns in {validation_path}'
    val['seed'] = val['seed'].astype(int)
    assert len(val) == 5 and set(val.seed) == SEEDS and val.seed.is_unique, f'Validation rows are not unique: {validation_path}'
    epochs = []
    for _, row in val.iterrows():
        history = list(log_dir.glob(f'training_history_{exp}_seed{int(row["seed"])}_*.json'))
        matching = [json.loads(p.read_text()) for p in history]
        matching = [h for h in matching if str(h.get('run_id')) == str(row['run_id'])]
        assert len(matching) == 1, f'Ambiguous or missing history for {row["run_id"]}'
        epochs.append(int(matching[0]['early_stopping']['best_epoch']))
    val['best_epoch'] = epochs
    cutoff = val.F1.max() - val.F1.std()
    selected = val[val.F1 >= cutoff].sort_values(['best_epoch', 'F1'], ascending=[True, False]).iloc[0]
    cached = references[exp]['test2'].iloc[0]
    selection.append(dict(experiment=exp, selected_seed=int(selected['seed']),
                          selected_run_id=str(selected['run_id']), cached_seed=int(cached['seed']),
                          cached_run_id=str(cached['run_id']),
                          agrees=str(selected['run_id']) == str(cached['run_id'])))
selection = pd.DataFrame(selection)
display(selection)
assert selection.agrees.all(), 'Cached champions do not match the validation 1-SD rule'


,model,seed,run_id,threshold_npz,threshold_cached_row,threshold_agrees,is_test2_champion
0,Simple MLP,42,SimpleMLP_BiasOn_seed42_20260219_181723,0.207085,0.207085,True,False
1,Simple MLP,123,SimpleMLP_BiasOn_seed123_20260219_183720,0.200319,0.200319,True,False
2,Simple MLP,777,SimpleMLP_BiasOn_seed777_20260219_185137,0.455980,0.455980,True,True
3,Simple MLP,2024,SimpleMLP_BiasOn_seed2024_20260219_193626,0.317028,0.317028,True,False
4,Simple MLP,99,SimpleMLP_BiasOn_seed99_20260219_202127,0.203490,0.203490,True,False
5,E-GraphSAGE,42,EGraphSAGE_BiasOn_seed42_20260421_181148,0.136257,0.136257,True,False
6,E-GraphSAGE,123,EGraphSAGE_BiasOn_seed123_20260421_184231,0.457199,0.457199,True,False
7,E-GraphSAGE,777,EGraphSAGE_BiasOn_seed777_20260421_192748,0.184951,0.184951,True,False
8,E-GraphSAGE,2024,EGraphSAGE_BiasOn_seed2024_20260421_195555,0.255702,0.255702,True,False
9,E-GraphSAGE,99,EGraphSAGE_BiasOn_seed99_20260421_201532,0.222804,0.222804,True,True


Identified checkpoints: 25
Differences between .npz threshold and CSV column: 0


,experiment,selected_seed,selected_run_id,cached_seed,cached_run_id,agrees
0,SimpleMLP_BiasOn,777,SimpleMLP_BiasOn_seed777_20260219_185137,777,SimpleMLP_BiasOn_seed777_20260219_185137,True
1,EGraphSAGE_BiasOn,99,EGraphSAGE_BiasOn_seed99_20260421_201532,99,EGraphSAGE_BiasOn_seed99_20260421_201532,True
2,EdgeGRU_NoX_BiasOn,99,EdgeGRU_NoX_BiasOn_seed99_20260419_205758,99,EdgeGRU_NoX_BiasOn_seed99_20260419_205758,True
3,StaticGNN_BiasOn_robust_Identity,42,StaticGNN_BiasOn_robust_Identity_seed42_202602...,42,StaticGNN_BiasOn_robust_Identity_seed42_202602...,True
4,ST_GNN_BiasOn_robust_Identity_clone,42,ST_GNN_BiasOn_robust_Identity_clone_seed42_202...,42,ST_GNN_BiasOn_robust_Identity_clone_seed42_202...,True


## 2. Independent inference on Test 1

Loads the **25 original checkpoints** and traverses the same Wednesday graphs in temporal order. If Drive access is slow, run a first partial pass with `AUDIT_EXPERIMENTS = ['ST_GNN_BiasOn_robust_Identity_clone']`, then run the full list. The paper summary requires all five models to be verified.


In [6]:
AUDIT_EXPERIMENTS = [exp for _, exp, _, _ in EXPERIMENTS]  # Change only for an initial partial pass.
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.0], device=DEVICE))

def replay(tag, experiments=AUDIT_EXPERIMENTS):
    root, split, _ = SPLITS[tag]
    dataset = NF_IDS_Dataset(root_dir=str(root), split=split)
    assert len(dataset) == EXPECTED_SNAPSHOTS[tag], f'Unexpected graph count in {root / split}: {len(dataset)}'
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    print(f'{tag}: {len(dataset)} graphs from {root / split}')
    records, label_digest = [], None
    for display_name, exp, model_class, temporal in EXPERIMENTS:
        if exp not in experiments:
            continue
        ref = references[exp][tag]
        for _, old in ref.iterrows():
            seed, run_id = int(old['seed']), str(old['run_id'])
            matched = inventory[(inventory.experiment == exp) & (inventory.run_id == run_id)]
            assert len(matched) == 1, f'No inventoried checkpoint for {run_id}'
            rec = matched.iloc[0]
            model = model_class(**BASE_PARAMS).to(DEVICE)
            state = torch.load(rec['checkpoint'], map_location=DEVICE, weights_only=True)
            model.load_state_dict(state, strict=True)
            model.eval()
            _, y, probs = infer(model, loader, criterion, DEVICE, is_temporal=temporal)
            y = np.asarray(y).astype(np.int8)
            digest = hashlib.sha256(y.tobytes()).hexdigest()
            if label_digest is None:
                label_digest = digest
            assert label_digest == digest, f'Flow order or labels differ across models in {tag}'
            metrics = calculate_metrics_gnn(y, probs, prob_threshold=float(rec['threshold_npz']))
            checks = {}
            for col in ['TP','FP','TN','FN']:
                checks[col] = int(metrics[col]) == int(old[col]) if col in old and pd.notna(old[col]) else None
            for col in ['Precision','Recall','F1','F2','AUC-PR','AUC-ROC','FPR']:
                checks[col] = bool(np.isclose(metrics[col], float(old[col]), rtol=1e-4, atol=1e-4)) if col in old and pd.notna(old[col]) else None
            comparable = [x for x in checks.values() if x is not None]
            status = 'MATCH' if comparable and all(comparable) else 'MISMATCH'
            records.append(dict(test=tag, model=display_name, seed=seed, run_id=run_id,
                                checkpoint_sha256=rec['checkpoint_sha256'], threshold=float(rec['threshold_npz']),
                                flows=len(y), attacks=int(y.sum()), labels_sha256=digest, status=status,
                                differences=','.join(k for k,v in checks.items() if v is False),
                                **{f'new_{k}':v for k,v in metrics.items()},
                                **{f'old_{k}':old.get(k, np.nan) for k in metrics}))
            print(f'{tag:5s} {display_name:12s} seed={seed:4d} {status:8s} '
                  f'F1 {metrics["F1"]:.4f} / {float(old["F1"]):.4f} '
                  f'AP {metrics["AUC-PR"]:.4f} / {float(old["AUC-PR"]):.4f} '
                  f'({records[-1]["differences"]})')
            del model, state
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return pd.DataFrame(records)

audit_t1 = replay('test1')
display(audit_t1[['model','seed','run_id','flows','attacks','status','differences','new_F1','old_F1','new_AUC-PR','old_AUC-PR']])


test1: 429 graphs from /content/drive/MyDrive/nids-mitre/dataset_processed/test
test1 Simple MLP   seed=  42 MATCH    F1 0.0010 / 0.0010 AP 0.0462 / 0.0462 ()
test1 Simple MLP   seed= 123 MATCH    F1 0.0042 / 0.0042 AP 0.0460 / 0.0460 ()
test1 Simple MLP   seed= 777 MATCH    F1 0.0000 / 0.0000 AP 0.0591 / 0.0591 ()
test1 Simple MLP   seed=2024 MATCH    F1 0.0000 / 0.0000 AP 0.0597 / 0.0597 ()
test1 Simple MLP   seed=  99 MATCH    F1 0.0010 / 0.0010 AP 0.0439 / 0.0439 ()
test1 E-GraphSAGE  seed=  42 MATCH    F1 0.4929 / 0.4929 AP 0.3257 / 0.3257 ()
test1 E-GraphSAGE  seed= 123 MATCH    F1 0.4941 / 0.4941 AP 0.3391 / 0.3391 ()
test1 E-GraphSAGE  seed= 777 MATCH    F1 0.4489 / 0.4489 AP 0.3549 / 0.3549 ()
test1 E-GraphSAGE  seed=2024 MATCH    F1 0.4878 / 0.4878 AP 0.3044 / 0.3044 ()
test1 E-GraphSAGE  seed=  99 MATCH    F1 0.4958 / 0.4958 AP 0.3290 / 0.3290 ()
test1 Edge GRU     seed=  42 MATCH    F1 0.4952 / 0.4952 AP 0.6317 / 0.6317 ()
test1 Edge GRU     seed= 123 MATCH    F1 0.1862 / 0

,model,seed,run_id,flows,attacks,status,differences,new_F1,old_F1,new_AUC-PR,old_AUC-PR
0,Simple MLP,42,SimpleMLP_BiasOn_seed42_20260219_181723,191679,3434,MATCH,,0.001018,0.001018,0.046250,0.046250
1,Simple MLP,123,SimpleMLP_BiasOn_seed123_20260219_183720,191679,3434,MATCH,,0.004163,0.004163,0.045965,0.045965
2,Simple MLP,777,SimpleMLP_BiasOn_seed777_20260219_185137,191679,3434,MATCH,,0.000000,0.000000,0.059103,0.059103
3,Simple MLP,2024,SimpleMLP_BiasOn_seed2024_20260219_193626,191679,3434,MATCH,,0.000000,0.000000,0.059746,0.059746
4,Simple MLP,99,SimpleMLP_BiasOn_seed99_20260219_202127,191679,3434,MATCH,,0.001026,0.001026,0.043851,0.043851
5,E-GraphSAGE,42,EGraphSAGE_BiasOn_seed42_20260421_181148,191679,3434,MATCH,,0.492854,0.492854,0.325745,0.325745
6,E-GraphSAGE,123,EGraphSAGE_BiasOn_seed123_20260421_184231,191679,3434,MATCH,,0.494126,0.494126,0.339058,0.339058
7,E-GraphSAGE,777,EGraphSAGE_BiasOn_seed777_20260421_192748,191679,3434,MATCH,,0.448902,0.448902,0.354887,0.354887
8,E-GraphSAGE,2024,EGraphSAGE_BiasOn_seed2024_20260421_195555,191679,3434,MATCH,,0.487843,0.487843,0.304378,0.304378
9,E-GraphSAGE,99,EGraphSAGE_BiasOn_seed99_20260421_201532,191679,3434,MATCH,,0.495818,0.495818,0.329013,0.329013


## 3. Independent inference on Test 2

Evaluates **one validation-selected champion per architecture** using the same checkpoints and thresholds. It loads all 2,822 Thursday graph files from `dataset_processed_thu0103/test2`. The 1,352 count in the latency section of `comparative_evaluation.ipynb` includes only nonempty graphs processed by inference; empty graphs are skipped here as well.


In [7]:
audit_t2 = replay('test2')
display(audit_t2[['model','seed','run_id','flows','attacks','status','differences','new_F1','old_F1','new_AUC-PR','old_AUC-PR']])


test2: 2822 graphs from /content/drive/MyDrive/nids-mitre/dataset_processed_thu0103/test2
test2 Simple MLP   seed= 777 MATCH    F1 0.4722 / 0.4722 AP 0.3961 / 0.3961 ()
test2 E-GraphSAGE  seed=  99 MATCH    F1 0.4950 / 0.4950 AP 0.5050 / 0.5050 ()
test2 Edge GRU     seed=  99 MATCH    F1 0.6502 / 0.6502 AP 0.7307 / 0.7307 ()
test2 Static GNN   seed=  42 MATCH    F1 0.2944 / 0.2944 AP 0.3142 / 0.3142 ()
test2 ST-GNN       seed=  42 MATCH    F1 0.7576 / 0.7576 AP 0.8384 / 0.8384 ()


,model,seed,run_id,flows,attacks,status,differences,new_F1,old_F1,new_AUC-PR,old_AUC-PR
0,Simple MLP,777,SimpleMLP_BiasOn_seed777_20260219_185137,2135957,103825,MATCH,,0.472171,0.472171,0.396098,0.396098
1,E-GraphSAGE,99,EGraphSAGE_BiasOn_seed99_20260421_201532,2135957,103825,MATCH,,0.494958,0.494958,0.504996,0.504996
2,Edge GRU,99,EdgeGRU_NoX_BiasOn_seed99_20260419_205758,2135957,103825,MATCH,,0.650233,0.650233,0.730656,0.730656
3,Static GNN,42,StaticGNN_BiasOn_robust_Identity_seed42_202602...,2135957,103825,MATCH,,0.294421,0.294421,0.314211,0.314211
4,ST-GNN,42,ST_GNN_BiasOn_robust_Identity_clone_seed42_202...,2135957,103825,MATCH,,0.757604,0.757604,0.838383,0.838383


## 4. Audit decision and record

`MATCH` means inference with the identified checkpoint and threshold reproduces the earlier CSV **for this code version and these graphs**. A mismatch requires investigating historical code, an overwritten threshold, regenerated graphs, or mixed CSVs. Results are not corrected automatically.


In [8]:
audit = pd.concat([audit_t1, audit_t2], ignore_index=True)
print('Verified evaluations (model × seed × test):', len(audit))
print(audit.groupby(['test','status']).size().to_string())
print('Test 1:')
print(audit_t1.groupby('model')[['new_F1','new_AUC-PR']].agg(['mean','std']).round(4).to_string())
print('Test 2:')
print(audit_t2.set_index('model')[['seed','new_F1','new_AUC-PR','new_Precision','new_Recall','new_FPR']].round(4).to_string())
if (audit.status != 'MATCH').any():
    print('REVIEW: discrepancies remain; do not cite earlier numbers as audited results yet.')
else:
    print('Earlier metrics were reproduced with the specified artifacts and code.')
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_dir = DRIVE_ROOT / 'results_comparison' / 'audit_original'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'audit_original_{stamp}.csv'
audit.to_csv(out_path, index=False)
inventory.to_csv(out_dir / f'inventory_original_{stamp}.csv', index=False)
selection.to_csv(out_dir / f'selection_original_{stamp}.csv', index=False)
print('Audit records:', out_dir, 'suffix', stamp)


Verified evaluations (model × seed × test): 30
test   status
test1  MATCH     25
test2  MATCH      5
Test 1:
             new_F1         new_AUC-PR        
               mean     std       mean     std
model                                         
E-GraphSAGE  0.4839  0.0198     0.3306  0.0185
Edge GRU     0.4289  0.1745     0.5143  0.2452
ST-GNN       0.7041  0.0319     0.6710  0.0156
Simple MLP   0.0012  0.0017     0.0510  0.0078
Static GNN   0.5674  0.0719     0.4257  0.0423
Test 2:
             seed  new_F1  new_AUC-PR  new_Precision  new_Recall  new_FPR
model                                                                    
Simple MLP    777  0.4722      0.3961         0.8805      0.3226   0.0022
E-GraphSAGE    99  0.4950      0.5050         0.5787      0.4324   0.0161
Edge GRU       99  0.6502      0.7307         0.5990      0.7110   0.0243
Static GNN     42  0.2944      0.3142         0.5094      0.2071   0.0102
ST-GNN         42  0.7576      0.8384         0.6336      0.941